In [29]:
import numpy as np
import os
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchvision import transforms
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [30]:
def preprocess(image_tensor):

    eps = 1e-8
    I = image_tensor + eps

    I_max = I.max()
    log_contrast = (torch.log(I_max/I))**2

    grad_y = torch.gradient(log_contrast, dim=-2)[0]  # height dimension
    grad_xy = torch.gradient(grad_y, dim=-1)[0]        # width dimension

    preprocessed = torch.abs(torch.tanh(grad_xy))

    return preprocessed

In [ ]:
class LensDataset(Dataset):

    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.paths = []
        self.labels = []

        # find classes
        self.classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])

        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        # collect files
        for cls in self.classes:
            cls_folder = os.path.join(root_dir, cls)

            for file in os.listdir(cls_folder):
                if file.endswith(".npy"):
                    self.paths.append(os.path.join(cls_folder, file))
                    self.labels.append(self.class_to_idx[cls])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):

        path = self.paths[idx]
        label = self.labels[idx]

        # load npy file
        image = np.load(path)

        # convert to tensor
        image = torch.tensor(image, dtype=torch.float32)
        physics = preprocess(image)

        combined = torch.cat([image,physics], dim = 0)

        if self.transform:
            image = self.transform(combined)

        return image, label

In [ ]:
train_dataset_no_norm = LensDataset(
    "dataset/train"
)

def compute_channel_stats(dataset):
    """
    Compute per-channel mean and std across the training set.
    Call this BEFORE creating your final DataLoader.
    """
    sum_ch = torch.zeros(2)
    sum_sq_ch = torch.zeros(2)
    count = 0
    
    # temporary loader with no normalization
    temp_loader = DataLoader(dataset, batch_size=64, 
                             shuffle=False, num_workers=4)
    
    for images, _ in temp_loader:
        # images shape: (B, 2, H, W)
        sum_ch += images.sum(dim=[0, 2, 3])
        sum_sq_ch += (images ** 2).sum(dim=[0, 2, 3])
        count += images.shape[0] * images.shape[2] * images.shape[3]
    
    mean = sum_ch / count
    std = torch.sqrt(sum_sq_ch / count - mean ** 2)
    
    return mean.tolist(), std.tolist()

# Run this once on your training set
# Then hardcode the results into your transforms
mean, std = compute_channel_stats(train_dataset_no_norm)
print("Channel means:", mean)   # e.g. [0.12, 0.04]
print("Channel stds:", std)     # e.g. [0.18, 0.06]

In [ ]:

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                     std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                     std=[0.229, 0.224, 0.225])
])


In [ ]:
train_dataset = LensDataset(
    "dataset/train",
    transform=train_transform
)

val_dataset = LensDataset(
    "dataset/val",
    transform=val_transform
)

In [25]:

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, 
                        num_workers=4, pin_memory=True)

In [26]:
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Train batches: 938
Val batches: 235


In [27]:
images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch labels:", labels)

Batch image shape: torch.Size([32, 3, 224, 224])
Batch labels: tensor([2, 2, 0, 1, 0, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 1, 2, 1, 2, 1, 0, 2, 1,
        2, 1, 1, 0, 2, 2, 1, 1])


In [28]:
from collections import Counter

train_counts = Counter(train_dataset.labels)
val_counts = Counter(val_dataset.labels)

print("Train distribution:", train_counts)
print("Val distribution:", val_counts)

Train distribution: Counter({0: 10000, 1: 10000, 2: 10000})
Val distribution: Counter({0: 2500, 1: 2500, 2: 2500})


In [ ]:
def preprocess(image_tensor):

    eps = 1e-8
    I = image_tensor + eps

    I_max = I.max()
    log_contrast = (torch.log(I_max/I))**2

    grad_y = torch.gradient(log_contrast, dim=-2)[0]  # height dimension
    grad_xy = torch.gradient(grad_y, dim=-1)[0]        # width dimension

    preprocessed = torch.abs(torch.tanh(grad_xy))

    return preprocessed